<a href="https://colab.research.google.com/github/anhndt0310-jpg/vietnamese-cyberbullying-detection/blob/main/vietnamese_cyberbullying_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers torch pandas numpy scikit-learn xgboost datasets tensorboard
!pip install -U accelerate
!pip install scikit-learn matplotlib

In [2]:
!pip install transformers==4.44.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 57.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.17.0
    Uninstalling huggingface_hub-1.17.0:
      Successfully uninstalled huggingface_hub-1.17.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.9.0
    Uninstalling transformers-5.9.0:
      Successfully uninstalled transformers-5.9.0


In [3]:
import os
import pandas as pd
# Trên RunPod, thư mục /workspace là nơi dữ liệu được giữ lại vĩnh viễn
if os.path.exists('/workspace'):
    base_path = '/workspace/vietnamese-cyberbullying-detection/'
    # Tạo thư mục nếu chưa có
    os.makedirs(base_path, exist_ok=True)
else:
    base_path = './'

print(f"📁 Dữ liệu sẽ được lưu tại: {base_path}")

📁 Dữ liệu sẽ được lưu tại: ./


In [4]:
import pandas as pd
import os

# --- CẤU HÌNH ĐƯỜNG DẪN ---
# Kiểm tra nếu là Google Colab
if 'google.colab' in str(get_ipython()):
    from google.colab import drive
    drive.mount('/content/drive')
    base_path = '/content/drive/MyDrive/vietnamese-cyberbullying-detection/'
# Mặc định là Codespaces (hoặc môi trường có file nằm cùng thư mục)
else:
    base_path = './'

# --- NẠP DỮ LIỆU ---
train_df = pd.read_csv(os.path.join(base_path, 'df_train_clean.csv'))
val_df = pd.read_csv(os.path.join(base_path, 'df_dev_clean.csv'))
test_df = pd.read_csv(os.path.join(base_path, 'df_test_clean.csv'))

print("✅ Đã tải dữ liệu thành công!")
print(f"Train: {len(train_df)} dòng | Dev: {len(val_df)} dòng | Test: {len(test_df)} dòng")

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import os
import numpy as np

# Tự động xác định đường dẫn lưu file
# Chúng ta dùng try-except để tránh lỗi khi không có thư viện google.colab
try:
    from google.colab import drive
    is_colab = True
except ImportError:
    is_colab = False

if is_colab:
    print("🚀 Đang chạy trên Google Colab...")
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    base_path = '/content/drive/MyDrive/vietnamese-cyberbullying-detection/'
else:
    print("💻 Đang chạy trên môi trường khác (Codespaces/Local)...")
    base_path = './'

# Tiếp tục các phần còn lại của bạn với base_path đã được xác định...
emb_train_path = os.path.join(base_path, 'X_train_emb.npy')
emb_test_path = os.path.join(base_path, 'X_test_emb.npy')
# 2. Quy trình nạp hoặc trích xuất
if os.path.exists(emb_train_path) and os.path.exists(emb_test_path):
    print("🔄 Đang nạp đặc trưng từ:", base_path)
    X_train_emb = np.load(emb_train_path)
    X_test_emb = np.load(emb_test_path)
    print("✅ Đã nạp xong!")
else:
    print("⚠️ Không tìm thấy file lưu sẵn. Đang bắt đầu trích xuất (có thể mất thời gian)...")
    # Đảm bảo các hàm này đã được định nghĩa trước đó
    X_train_emb = get_phobert_embeddings(train_df['content_clean'].fillna('').tolist())
    X_test_emb = get_phobert_embeddings(test_df['content_clean'].fillna('').tolist())

    # Lưu file
    np.save(emb_train_path, X_train_emb)
    np.save(emb_test_path, X_test_emb)
    print(f"✅ Đã trích xuất và lưu vào {base_path}!")

y_train = train_df['Toxicity'].values
y_test = test_df['Toxicity'].values

print(f"Kích thước tập Train: {X_train_emb.shape}")
print(f"Kích thước tập Test: {X_test_emb.shape}")

In [ ]:
# Xác định base_path dựa trên môi trường (đã làm ở bước trước)
# Nếu bạn đã có biến base_path, hãy dùng luôn nó nhé.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report
import joblib
import os
save_path = os.path.join(base_path, 'phobert_logreg_model.pkl')

print(f"Đang huấn luyện Logistic Regression...")
lr_final = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)
lr_final.fit(X_train_emb, y_train)

# Lưu mô hình vào đường dẫn linh hoạt
joblib.dump(lr_final, save_path)
print(f"✅ Đã lưu mô hình tại: {save_path}")

# Dự đoán và đánh giá
y_pred_lr = lr_final.predict(X_test_emb)

print("\n=== KẾT QUẢ LOGISTIC REGRESSION ===")
print(confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

In [ ]:
import os
import joblib
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Đảm bảo base_path đã được định nghĩa như chúng ta đã làm trước đó
# Ví dụ: base_path = '/workspace/vietnamese-cyberbullying-detection/'
# --- CẤU HÌNH PATH (Dựa trên biến base_path đã định nghĩa từ trước) ---
rf_model_path = os.path.join(base_path, 'rf_model.pkl')
xgb_model_path = os.path.join(base_path, 'xgb_model.pkl')

# --- 1. HUẤN LUYỆN HOẶC NẠP RANDOM FOREST ---
if os.path.exists(rf_model_path):
    print("🔄 Đang nạp mô hình Random Forest từ file...")
    rf_model = joblib.load(rf_model_path)
else:
    print("🚀 Đang huấn luyện Random Forest...")
    rf_model = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
    rf_model.fit(X_train_emb, y_train)
    joblib.dump(rf_model, rf_model_path)
    print("✅ Đã lưu Random Forest!")

# --- 2. HUẤN LUYỆN HOẶC NẠP XGBOOST ---
if os.path.exists(xgb_model_path):
    print("🔄 Đang nạp mô hình XGBoost từ file...")
    xgb_model = joblib.load(xgb_model_path)
else:
    print("🚀 Đang huấn luyện XGBoost...")
    # Tự động tính tỉ lệ scale_pos_weight nếu là bài toán nhị phân
    scale_weight = (y_train == 0).sum() / (y_train == 1).sum() if 1 in y_train else 1

    xgb_model = XGBClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=6,
        scale_pos_weight=scale_weight, eval_metric='logloss', random_state=42
    )
    xgb_model.fit(X_train_emb, y_train)
    joblib.dump(xgb_model, xgb_model_path)
    print("✅ Đã lưu XGBoost!")

# --- DỰ ĐOÁN ---
y_pred_rf = rf_model.predict(X_test_emb)
y_pred_xgb = xgb_model.predict(X_test_emb)

print("\n=== KẾT QUẢ SO SÁNH ===")
print("Random Forest Accuracy:", rf_model.score(X_test_emb, y_test))
print("XGBoost Accuracy:", xgb_model.score(X_test_emb, y_test))

In [ ]:
import os
import pandas as pd
from datasets import Dataset, load_from_disk
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, AutoTokenizer
from sklearn.metrics import accuracy_score, f1_score

# --- CẤU HÌNH ---
TRAIN_MODE = True  # Đổi thành False nếu bạn chỉ muốn nạp model để dự đoán
base_path = os.getcwd()
data_dir = os.path.join(base_path, 'data')
results_dir = os.path.join(base_path, 'results')

os.makedirs(data_dir, exist_ok=True)

# 1. LOAD DỮ LIỆU (Đã tối ưu cache)
def get_data():
    data_path_train = os.path.join(data_dir, 'train_tokenized')
    if os.path.exists(data_path_train):
        print("🚀 Đã tìm thấy dữ liệu đã tokenize! Đang nạp từ ổ cứng...")
        return load_from_disk(data_path_train), load_from_disk(os.path.join(data_dir, 'test_tokenized'))
    else:
        # Giả sử train_df, test_df đã được load trước đó
        print("⏳ Không thấy dữ liệu, đang thực hiện tokenize...")
        tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")
        train_ds = Dataset.from_pandas(train_df[['content_clean', 'Toxicity']].rename(columns={'content_clean': 'text', 'Toxicity': 'label'}))
        test_ds = Dataset.from_pandas(test_df[['content_clean', 'Toxicity']].rename(columns={'content_clean': 'text', 'Toxicity': 'label'}))

        train_ds = train_ds.map(lambda x: tokenizer(x['text'], padding='max_length', truncation=True, max_length=128), batched=True)
        test_ds = test_ds.map(lambda x: tokenizer(x['text'], padding='max_length', truncation=True, max_length=128), batched=True)

        train_ds.save_to_disk(data_path_train)
        test_ds.save_to_disk(os.path.join(data_dir, 'test_tokenized'))
        return train_ds, test_ds

train_dataset, test_dataset = get_data()

# 2. LOAD MÔ HÌNH (Tự động chọn: Model gốc hoặc Model đã train)
def load_model():
    # Kiểm tra xem có checkpoint nào đã lưu chưa
    checkpoints = [os.path.join(results_dir, d) for d in os.listdir(results_dir) if d.startswith("checkpoint")]
    if checkpoints and not TRAIN_MODE:
        latest_checkpoint = max(checkpoints, key=os.path.getctime)
        print(f"📦 Nạp mô hình đã train từ: {latest_checkpoint}")
        return AutoModelForSequenceClassification.from_pretrained(latest_checkpoint)
    else:
        print("🆕 Nạp mô hình mới (vinai/phobert-base)...")
        return AutoModelForSequenceClassification.from_pretrained("vinai/phobert-base", num_labels=2)

model = load_model()

# 3. HUẤN LUYỆN
if TRAIN_MODE:
    training_args = TrainingArguments(
        output_dir=results_dir,
        num_train_epochs=3,
        per_device_train_batch_size=16,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        fp16=True,
        report_to="tensorboard"
    )

    trainer = Trainer(
        model=model, args=training_args,
        train_dataset=train_dataset, eval_dataset=test_dataset,
        compute_metrics=lambda p: {'accuracy': accuracy_score(p.label_ids, p.predictions.argmax(-1)),
                                   'f1': f1_score(p.label_ids, p.predictions.argmax(-1), average='weighted')}
    )

    print("🚀 Đang bắt đầu huấn luyện...")
    trainer.train()
    print("✅ Huấn luyện hoàn tất!")
else:
    print("🔍 Chế độ dự đoán (Inference). Mô hình đã sẵn sàng.")

In [ ]:
# 1. Lưu mô hình sau khi Fine-tuning
model_save_path = '/content/drive/MyDrive/phobert_finetuned_toxicity'
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"✅ Đã lưu mô hình fine-tuned tại: {model_save_path}")

# 2. Đánh giá chi tiết trên tập test
evaluations = trainer.evaluate()
print("\n=== KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST ===")
for key, value in evaluations.items():
    print(f"{key}: {value}")

2. Thêm Các Metric Đánh giá Chi tiết

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, precision_recall_curve, average_precision_score,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
import numpy as np

def evaluate_model(y_true, y_pred, y_proba=None, model_name="Model"):
    print(f"\n{'='*50}")
    print(f"KẾT QUẢ ĐÁNH GIÁ: {model_name}")
    print('='*50)

    # Classification Report
    print("\n📊 Classification Report:")
    print(classification_report(y_true, y_pred, target_names=['Non-Toxic', 'Toxic']))

    # Confusion Matrix
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=['Non-Toxic', 'Toxic'], ax=ax, cmap='Blues')
    plt.title(f'Confusion Matrix - {model_name}')
    plt.tight_layout()
    plt.show()

    # ROC-AUC nếu có probability
    if y_proba is not None:
        auc = roc_auc_score(y_true, y_proba)
        ap = average_precision_score(y_true, y_proba)
        print(f"\n🎯 ROC-AUC Score: {auc:.4f}")
        print(f"🎯 Average Precision: {ap:.4f}")

# Sử dụng các biến dự đoán đã có sẵn từ cell JgyecsOHMYWz
try:
    # Nếu logreg_model chưa được định nghĩa, ta dùng biến 'model' từ cell huấn luyện Logistic
    y_proba_logreg = None
    if 'model' in globals() and hasattr(model, 'predict_proba'):
        y_proba_logreg = model.predict_proba(X_test_emb)[:, 1]

    evaluate_model(y_test, y_pred, y_proba_logreg, "PhoBERT + Logistic Regression")
except Exception as e:
    print(f"Có lỗi xảy ra khi đánh giá: {e}")
    print("Vui lòng đảm bảo bạn đã chạy cell huấn luyện Logistic Regression trước đó.")

3. Cross-Validation để đánh giá ổn định

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Cross-validation scores
cv_scores = cross_val_score(
    LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42),
    X_train_emb, y_train,
    cv=cv, scoring='f1_weighted', n_jobs=-1
)

print(f"Cross-Validation F1 Scores: {cv_scores}")
print(f"Mean F1: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")


4. Lưu và Tải Model

In [ ]:
import joblib

# Lưu model
joblib.dump(model, '/content/drive/MyDrive/phobert_logreg_model.pkl')

# Lưu embeddings (để không phải chạy lại)
np.save('/content/drive/MyDrive/X_train_emb.npy', X_train_emb)
np.save('/content/drive/MyDrive/X_test_emb.npy', X_test_emb)

# Tải lại
model = joblib.load('/content/drive/MyDrive/phobert_logreg_model.pkl')
X_train_emb = np.load('/content/drive/MyDrive/X_train_emb.npy')


5. Hàm Dự đoán cho câu mới

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
import joblib
import os

# 1. Sử dụng lại các biến đã có để tránh tải lại model
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Kiểm tra nếu chưa có model/tokenizer trong bộ nhớ thì mới load
if 'phobert' not in globals():
    print("📦 Đang nạp PhoBERT base model (lần đầu)...")
    phobert_base = AutoModel.from_pretrained("vinai/phobert-base").to(device)
else:
    phobert_base = phobert # Dùng lại biến phobert từ cell 14b8be58

if 'tokenizer' not in globals():
    tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")

# 2. Tải mô hình Logistic Regression đã huấn luyện
model_path = '/content/drive/MyDrive/phobert_logreg_model.pkl'
if os.path.exists(model_path):
    final_ml_model = joblib.load(model_path)
else:
    final_ml_model = lr_final # Dùng trực tiếp nếu file chưa kịp lưu

def predict_toxicity_ml(text, ml_model, tokenizer, embed_model):
    embed_model.eval()
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)

    with torch.no_grad():
        outputs = embed_model(**inputs)

    # Lấy vector CLS
    embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy()

    # Dự đoán
    prediction = ml_model.predict(embedding)[0]
    probability = ml_model.predict_proba(embedding)[0]

    result = "🚨 ĐỘC HẠI" if prediction == 1 else "✅ KHÔNG ĐỘC HẠI"
    print(f"Văn bản: {text}")
    print(f"Kết quả: {result} ({probability[1]:.2%} toxic)")
    print("-" * 30)

print("--- THỬ NGHIỆM DỰ ĐOÁN NHANH ---")
predict_toxicity_ml("Bạn thật tuyệt vời!", final_ml_model, tokenizer, phobert_base)
predict_toxicity_ml("Đồ ngu ngốc vô dụng!", final_ml_model, tokenizer, phobert_base)

6. So sánh nhiều Model (Tổng hợp)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1),
    'SVM': SVC(class_weight='balanced', probability=True, random_state=42),
    'MLP Neural Network': MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=500, random_state=42),
}

results = []
for name, clf in models.items():
    print(f"\n🔄 Đang huấn luyện {name}...")
    clf.fit(X_train_emb, y_train)
    y_pred = clf.predict(X_test_emb)

    from sklearn.metrics import f1_score, accuracy_score
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'F1 (Weighted)': f1_score(y_test, y_pred, average='weighted'),
        'F1 (Toxic)': f1_score(y_test, y_pred, pos_label=1)
    })

results_df = pd.DataFrame(results).sort_values('F1 (Weighted)', ascending=False)
print("\n" + "="*60)
print("📊 BẢNG SO SÁNH CÁC MÔ HÌNH")
print("="*60)
print(results_df.to_string(index=False))


In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, f1_score
from sklearn.ensemble import RandomForestClassifier
from tqdm import tqdm

# 1. Đảm bảo các thành phần PhoBERT đã sẵn sàng
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")
phobert = AutoModel.from_pretrained("vinai/phobert-base").to(device)

def get_phobert_embeddings(texts, batch_size=16):
    phobert.eval()
    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i+batch_size]
        inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
        with torch.no_grad():
            outputs = phobert(**inputs)
        # Lấy vector [CLS]
        batch_emb = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        embeddings.append(batch_emb)
    return np.vstack(embeddings)

# 2. Kiểm tra và trích xuất lại đặc trưng
if 'X_train_emb' not in globals():
    print("🔄 Đang trích xuất đặc trưng (có thể mất ít phút)...")
    X_train_emb = get_phobert_embeddings(train_df['content_clean'].fillna('').tolist())
    X_test_emb = get_phobert_embeddings(test_df['content_clean'].fillna('').tolist())
    y_train = train_df['Toxicity'].values
    y_test = test_df['Toxicity'].values

print("--- TỐI ƯU RANDOM FOREST (THRESHOLD MOVING) ---")

# 3. Huấn luyện với trọng số cực cao cho lớp Toxic
rf_optimized = RandomForestClassifier(n_estimators=300,
                                      class_weight={0: 1, 1: 12},
                                      max_depth=20,
                                      random_state=42,
                                      n_jobs=-1)

print("Đang huấn luyện Random Forest...")
rf_optimized.fit(X_train_emb, y_train)

# 4. Sử dụng Threshold Moving (Hạ xuống 0.25 để ưu tiên Recall)
y_probs_rf = rf_optimized.predict_proba(X_test_emb)[:, 1]
threshold = 0.25
y_pred_rf_custom = (y_probs_rf >= threshold).astype(int)

print(f"\nKết quả Random Forest sau khi chỉnh ngưỡng {threshold}:")
print(classification_report(y_test, y_pred_rf_custom, target_names=['Non-Toxic', 'Toxic']))

f1_toxic = f1_score(y_test, y_pred_rf_custom)
print(f"=> F1-score mới cho lớp Toxic: {f1_toxic:.4f}")